In [12]:
%%R

K_PATH <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/K_ppi_2_ME84_n5.csv"
OUT_DIR <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts"

# Read K as Hermitian matrix
K_df <- read.csv(K_PATH, header = FALSE, check.names = FALSE)

# Remove index column if present
if (ncol(K_df) == nrow(K_df) + 1) {
  K_df <- K_df[, -1]
}

K <- as.matrix(K_df)
storage.mode(K) <- "numeric"
K <- (K + Conj(t(K))) / 2

cat("K dim:", dim(K), "\n")
cat("trace:", sum(diag(K)), "\n")
cat("projection error:", max(abs(K %*% K - K)), "\n")

# ------------------------------------------------------------
# Patch your Reciprocal_CaDsd only for NA rho problem
# ------------------------------------------------------------
fun_txt <- deparse(Reciprocal_CaDsd)

fun_txt <- gsub(
  "rho\\[, k - 1\\] = Arg\\(\\(diag\\(V\\)\\)\\)",
  "dV <- diag(V); dV[is.na(dV)] <- 1+0i; rho[, k - 1] = Arg(dV)",
  fun_txt
)

fun_txt <- gsub(
  "rho\\[rho < 0\\] <- rho\\[rho < 0\\] \\+ 1",
  "rho[is.na(rho) | !is.finite(rho)] <- 0; rho[rho < 0] <- rho[rho < 0] + 1",
  fun_txt
)

# Also return U, because CaDsd may need the same initial U
fun_txt <- gsub(
  "list_data=list\\(K,omega,rho,mat_spectre, MatV\\)",
  "list_data=list(K,omega,rho,mat_spectre,MatV,U)",
  fun_txt
)

fun_txt <- gsub(
  'names\\(list_data\\)=c\\("KSort","omega","rho","spectre","MatV"\\)',
  'names(list_data)=c("KSort","omega","rho","spectre","MatV","U")',
  fun_txt
)

Reciprocal_CaDsd_safe <- eval(parse(text = paste(fun_txt, collapse = "\n")))

# Run reciprocal
res <- Reciprocal_CaDsd_safe(K)

KSort <- res$KSort
omega <- res$omega
rho <- res$rho
spectre <- res$spectre
U_seed <- res$U
pi_sorted <- Re(diag(KSort))

cat("omega dim:", dim(omega), "\n")
cat("rho dim:", dim(rho), "\n")
cat("NA omega:", sum(is.na(omega)), "\n")
cat("NA rho:", sum(is.na(rho)), "\n")
cat("pi_sorted sum:", sum(pi_sorted), "\n")

# Save outputs
write.csv(omega, file.path(OUT_DIR, "omega_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(rho, file.path(OUT_DIR, "rho_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(Re(KSort), file.path(OUT_DIR, "KSort_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(pi_sorted, file.path(OUT_DIR, "pi_sorted_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)

saveRDS(res, file.path(OUT_DIR, "reciprocal_from_K_ppi_2_ME84_n5_safe.rds"))

cat("Saved omega, rho, KSort, pi_sorted, and full reciprocal object.\n")

K dim: 281 281 
trace: 5 
projection error: 2.220446e-16 
omega dim: 5 281 
rho dim: 5 280 
NA omega: 0 
NA rho: 0 
pi_sorted sum: 5 
Saved omega, rho, KSort, pi_sorted, and full reciprocal object.


In [5]:
%%R

K_PATH <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/K_ppi_2_ME84_n5.csv"
OUT_DIR <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts"

cat("K exists:", file.exists(K_PATH), "\n")

# Source Vincent GitHub R codes directly
source("https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/CaDsd")
source("https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/Reciprocal_CaDsd")

cat("Functions loaded:\n")
cat("CaDsd:", exists("CaDsd"), "\n")
cat("Reciprocal_CaDsd:", exists("Reciprocal_CaDsd"), "\n")

# Read K matrix
K_df <- read.csv(K_PATH, header = FALSE, check.names = FALSE)

# Remove index column if present
if (ncol(K_df) == nrow(K_df) + 1) {
  K_df <- K_df[, -1]
}

K <- as.matrix(K_df)
storage.mode(K) <- "numeric"

# Force Hermitian / symmetric numerical form
K <- (K + Conj(t(K))) / 2

cat("K dimension:", dim(K), "\n")
cat("trace(K):", sum(diag(K)), "\n")
cat("diag range:", range(diag(K)), "\n")
cat("projection error:", max(abs(K %*% K - K)), "\n")

# Reciprocal extraction
res <- Reciprocal_CaDsd(K)

cat("Returned names:\n")
print(names(res))

omega <- res$omega
rho <- res$rho
KSort <- res$KSort
pi_sorted <- Re(diag(KSort))

cat("omega dim:", dim(omega), "\n")
cat("rho dim:", dim(rho), "\n")
cat("KSort dim:", dim(KSort), "\n")
cat("pi_sorted sum:", sum(pi_sorted), "\n")

# Save outputs
write.csv(omega, file.path(OUT_DIR, "omega_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(rho, file.path(OUT_DIR, "rho_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(Re(KSort), file.path(OUT_DIR, "KSort_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(pi_sorted, file.path(OUT_DIR, "pi_sorted_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)

saveRDS(res, file.path(OUT_DIR, "reciprocal_from_K_ppi_2_ME84_n5.rds"))

cat("Saved reciprocal outputs in:", OUT_DIR, "\n")

K exists: TRUE 
          [,1]      [,2]      [,3]      [,4]      [,5]      [,6]      [,7]
[1,] 0.0000000 0.0000000 0.0000000 0.0000000 0.0000000 0.3181818 0.3818182
[2,] 0.0000000 0.0000000 0.0000000 0.0000000 0.3818182 0.3818182 0.4454545
[3,] 0.0000000 0.0000000 0.0000000 0.4454545 0.4454545 0.4454545 0.5090909
[4,] 0.0000000 0.0000000 0.5090909 0.5090909 0.5090909 0.5090909 0.5727273
[5,] 0.0000000 0.5727273 0.5727273 0.5727273 0.5727273 0.5727273 0.5727273
[6,] 0.6363636 0.6363636 0.6363636 0.6363636 0.6363636 0.6363636 0.6363636
          [,8]      [,9]     [,10]
[1,] 0.4454545 0.5090909 0.5727273
[2,] 0.5090909 0.5727273 0.5727273
[3,] 0.5727273 0.5727273 0.5727273
[4,] 0.5727273 0.5727273 0.5727273
[5,] 0.5727273 0.5727273 0.5727273
[6,] 0.6363636 0.6363636 0.6363636
Functions loaded:
CaDsd: TRUE 
Reciprocal_CaDsd: TRUE 
K dimension: 281 281 
trace(K): 5 
diag range: 0.002933412 0.1012027 
projection error: 2.220446e-16 
Error in rho[rho < 0] <- rho[rho < 0] + 1 : 
  NAs are no

RInterpreterError: Failed to parse and evaluate line '\nK_PATH <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts/K_ppi_2_ME84_n5.csv"\nOUT_DIR <- "/home/bardia/projects/graphical-sampling/simulations_abc/jupyters/artifacts"\n\ncat("K exists:", file.exists(K_PATH), "\\n")\n\n# Source Vincent GitHub R codes directly\nsource("https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/CaDsd")\nsource("https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/Reciprocal_CaDsd")\n\ncat("Functions loaded:\\n")\ncat("CaDsd:", exists("CaDsd"), "\\n")\ncat("Reciprocal_CaDsd:", exists("Reciprocal_CaDsd"), "\\n")\n\n# Read K matrix\nK_df <- read.csv(K_PATH, header = FALSE, check.names = FALSE)\n\n# Remove index column if present\nif (ncol(K_df) == nrow(K_df) + 1) {\n  K_df <- K_df[, -1]\n}\n\nK <- as.matrix(K_df)\nstorage.mode(K) <- "numeric"\n\n# Force Hermitian / symmetric numerical form\nK <- (K + Conj(t(K))) / 2\n\ncat(

In [6]:
%%R

cat("Reciprocal_CaDsd exists:", exists("Reciprocal_CaDsd"), "\n")

fun_txt <- deparse(Reciprocal_CaDsd)

rho_lines <- grep("rho|acos|asin|Arg|angle|atan", fun_txt, ignore.case = TRUE)

cat("Lines related to rho/angles:\n")
for (i in rho_lines) {
  cat(i, ":", fun_txt[i], "\n")
}

Reciprocal_CaDsd exists: TRUE 
Lines related to rho/angles:
3 :     K <- K[order(diag(K), decreasing = TRUE), order(diag(K),  
4 :         decreasing = TRUE)] 
6 :     Pi = Re(diag(K)[order(diag(K), decreasing = TRUE)]) 
17 :     rho = matrix(0, M, N - 1) 
70 :         rho[, k - 1] = Arg((diag(V))) 
172 :     rho = round(rho/(2 * 3.141593), 7) 
173 :     rho[rho < 0] <- rho[rho < 0] + 1 
175 :     list_data = list(K, omega, rho, mat_spectre, MatV) 
176 :     names(list_data) = c("KSort", "omega", "rho", "spectre",  


In [7]:
%%R

fun_txt <- deparse(Reciprocal_CaDsd)

cat("Lines 55 to 80:\n")
for (i in 55:80) {
  cat(i, ":", fun_txt[i], "\n")
}

cat("\nLines 165 to 176:\n")
for (i in 165:176) {
  cat(i, ":", fun_txt[i], "\n")
}

Lines 55 to 80:
55 :                 else { 
56 :                   X = U_chap[, 0:(prev_pos - 1)] 
57 :                 } 
58 :                 for (j in 1:(occ[i] - 1)) { 
59 :                   X = cbind(U_chap[, (prev_pos + j - 1)], X) 
60 :                   U_chap[, prev_pos + j] = (diag(M) - X %*% solve(t(Conj(X)) %*%  
61 :                     X) %*% t(Conj(X))) %*% matrix(rnorm(M), M,  
62 :                     1) 
63 :                   U_chap[, prev_pos + j] = U_chap[, prev_pos +  
64 :                     j]/cnorm(U_chap[, prev_pos + j]) 
65 :                 } 
66 :             } 
67 :         } 
68 :         V = t(Conj(U)) %*% U_chap 
69 :         MatV[[k - 1]] <- V 
70 :         rho[, k - 1] = Arg((diag(V))) 
71 :         lambda1 = matrix(0, M, 1) 
72 :         lambda1[1:min(k, M)] = eigen(K[1:k, 1:k])$value[1:min(k,  
73 :             M)] 
74 :         lambda1 = round(rev(lambda1), 7) 
75 :         mat_spectre = cbind(mat_spectre, lambda1) 
76 :         ens = 1:M 
77 : 

In [8]:
%%R

# ============================================================
# Safe patched version of Vincent's Reciprocal_CaDsd
# ============================================================

fun_txt <- deparse(Reciprocal_CaDsd)

# Patch the line where rho is computed from Arg(diag(V))
fun_txt <- gsub(
  "rho\\[, k - 1\\] = Arg\\(\\(diag\\(V\\)\\)\\)",
  "dV <- diag(V); dV[!is.finite(Re(dV)) | !is.finite(Im(dV))] <- 1+0i; rho[, k - 1] = Arg(dV)",
  fun_txt
)

# Patch final rho correction
fun_txt <- gsub(
  "rho\\[rho < 0\\] <- rho\\[rho < 0\\] \\+ 1",
  "rho[is.na(rho) | !is.finite(rho)] <- 0; rho[rho < 0] <- rho[rho < 0] + 1",
  fun_txt
)

Reciprocal_CaDsd_safe <- eval(parse(text = paste(fun_txt, collapse = "\n")))

cat("Safe reciprocal function created:", exists("Reciprocal_CaDsd_safe"), "\n")

# Run safe reciprocal
res <- Reciprocal_CaDsd_safe(K)

cat("Returned names:\n")
print(names(res))

omega <- res$omega
rho <- res$rho
KSort <- res$KSort
pi_sorted <- Re(diag(KSort))

cat("omega dim:", dim(omega), "\n")
cat("rho dim:", dim(rho), "\n")
cat("NA in omega:", sum(is.na(omega)), "\n")
cat("NA in rho:", sum(is.na(rho)), "\n")
cat("rho range:", range(rho, na.rm = TRUE), "\n")

# Save outputs
write.csv(omega, file.path(OUT_DIR, "omega_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(rho, file.path(OUT_DIR, "rho_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(Re(KSort), file.path(OUT_DIR, "KSort_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)
write.csv(pi_sorted, file.path(OUT_DIR, "pi_sorted_from_K_ppi_2_ME84_n5.csv"), row.names = FALSE)

saveRDS(res, file.path(OUT_DIR, "reciprocal_from_K_ppi_2_ME84_n5_safe.rds"))

cat("Saved outputs in:", OUT_DIR, "\n")

Safe reciprocal function created: TRUE 
Returned names:
[1] "KSort"   "omega"   "rho"     "spectre" "MatV"   
omega dim: 5 281 
rho dim: 5 280 
NA in omega: 0 
NA in rho: 0 
rho range: 0 0.4999999 
Saved outputs in: /home/bardia/projects/graphical-sampling/simulations_abc/jupyters 


In [9]:
%%R

# ============================================================
# Check reconstruction: CaDsd(pi_sorted, omega, rho) -> KSort
# ============================================================

M <- nrow(omega)

cat("M:", M, "\n")
cat("length(pi_sorted):", length(pi_sorted), "\n")
cat("omega dim:", dim(omega), "\n")
cat("rho dim:", dim(rho), "\n")

# Reconstruct kernel from reciprocal parameters
rebuilt <- CaDsd(pi = pi_sorted, M = M, omega = omega, rho = rho)

# CaDsd may return either a matrix or a list containing K
if (is.list(rebuilt) && "K" %in% names(rebuilt)) {
  K_rebuilt <- rebuilt$K
} else {
  K_rebuilt <- rebuilt
}

K_rebuilt <- as.matrix(K_rebuilt)

cat("K_rebuilt dim:", dim(K_rebuilt), "\n")
cat("diag error:", max(abs(diag(K_rebuilt) - pi_sorted)), "\n")
cat("projection error:", max(abs(K_rebuilt %*% K_rebuilt - K_rebuilt)), "\n")
cat("reconstruction error vs KSort:", max(abs(K_rebuilt - KSort)), "\n")
cat("trace rebuilt:", sum(diag(K_rebuilt)), "\n")

write.csv(Re(K_rebuilt), file.path(OUT_DIR, "K_rebuilt_from_reciprocal_ME84_n5.csv"), row.names = FALSE)
cat("Saved rebuilt K.\n")

M: 5 
length(pi_sorted): 281 
omega dim: 5 281 
rho dim: 5 280 
K_rebuilt dim: 281 281 
diag error: 4.119472e-05 
projection error: 4.01022e-05 
reconstruction error vs KSort: 0.1385464 
trace rebuilt: 4.999093+0i 
Saved rebuilt K.


In [10]:
%%R

cat("Reciprocal_CaDsd first 25 lines:\n")
fun_txt <- deparse(Reciprocal_CaDsd)
for (i in 1:25) {
  cat(i, ":", fun_txt[i], "\n")
}

cat("\nCaDsd arguments:\n")
print(names(formals(CaDsd)))

cat("\nCaDsd first 25 lines:\n")
cad_txt <- deparse(CaDsd)
for (i in 1:25) {
  cat(i, ":", cad_txt[i], "\n")
}

Reciprocal_CaDsd first 25 lines:
1 : function (K)  
2 : { 
3 :     K <- K[order(diag(K), decreasing = TRUE), order(diag(K),  
4 :         decreasing = TRUE)] 
5 :     mu = Re(sum(diag(K))) 
6 :     Pi = Re(diag(K)[order(diag(K), decreasing = TRUE)]) 
7 :     N = length(Pi) 
8 :     spectre = eigen(K)$values 
9 :     M = match(0, round(spectre, 1)) - 1 
10 :     if (is.na(M)) { 
11 :         M = N 
12 :     } 
13 :     phi = Conj(t(eigen(K)$vectors %*% diag(sqrt(abs(spectre)))))[1:M,  
14 :         ] 
15 :     U_chap = matrix(0, M, M) 
16 :     omega = matrix(0, M, N) 
17 :     rho = matrix(0, M, N - 1) 
18 :     U = matrix(phi[, 1]/Pi[1]) 
19 :     Y = matrix(runif(M * (M - 1), 0, 1000), M, M - 1) 
20 :     for (j in 2:M) { 
21 :         vect = (diag(M) - U %*% diag(1/diag(t(Conj(U)) %*% U),  
22 :             j - 1) %*% t(Conj(U))) %*% Y[, j - 1] 
23 :         U = cbind(U, vect) 
24 :     } 
25 :     U = t(t(U)/sqrt(Re(colSums(Conj(U) * U)))) 

CaDsd arguments:
[1] "omega"   "rho"    